# Residual Scaling Experiment (RRE v1.0)

**Research question:** Can a more optimisation-friendly scaling of the *same* Prophet residual improve Hybrid GRU learning without changing underlying temporal information?

> This is **not** a temporal representation experiment. Stage 0 showed R3 (median/MAD) preserves identical ACF structure to R0 while improving numerical robustness.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

REPO_ROOT = Path('..').resolve()
sys.path.insert(0, str(REPO_ROOT))

EXP_DIR = REPO_ROOT / 'experiments/rre_2026-07-27_123441'
CONFIG = json.loads((EXP_DIR / 'config' / 'rre_config_frozen.json').read_text())
FINAL = json.loads((EXP_DIR / 'reports' / 'final_report.json').read_text())
COMP = json.loads((EXP_DIR / 'comparison' / 'paired_tests.json').read_text())
ANSWERS = FINAL['answers']

eval_r0 = pd.read_csv(EXP_DIR / 'variants' / 'R0' / 'evaluation' / 'evaluation_extended.csv')
eval_r3 = pd.read_csv(EXP_DIR / 'variants' / 'R3' / 'evaluation' / 'evaluation_extended.csv')
hist_r0 = pd.read_csv(EXP_DIR / 'variants' / 'R0' / 'training' / 'training_history.csv')
hist_r3 = pd.read_csv(EXP_DIR / 'variants' / 'R3' / 'training' / 'training_history.csv')

print('Protocol:', CONFIG['protocol_version'])
print('Research question:', CONFIG['research_question'])

## 2. Motivation and Stage 0 summary

Prior work established that Prophet explains ~78% of CPU variance, Hybrid beats Prophet, and changes to loss/context/windows did not help. **Stage 0** compared residual representations statistically:

- **R0** (level z-score): strongest temporal structure
- **R1** (velocity): rejected — removes temporal dependence
- **R2**: no clear advantage
- **R3** (median/MAD): **same ACF as R0**, lower sparsity, more robust tails

Therefore RRE v1.0 tests **optimisation conditioning**, not new temporal information.

## 3. Mathematical formulation

Level Prophet residual: $r_t = y_t - \hat{y}_t$

**R0 (baseline):** $\tilde{r}_t = (r_t - \mu_{train}) / \sigma_{train}$

**R3 (robust):** $\tilde{r}_t = (r_t - \mathrm{median}_{train}) / \mathrm{MAD}_{train}$

Inverse at inference: $\hat{r}_t = \tilde{r}_t \cdot s + c$ where $(c,s)$ are train-fitted centre and scale. GRU architecture, window, optimizer, and loss are frozen.

## 4. Cohort comparison table

In [ ]:
summary_rows = []
for vid in ['R0', 'R3']:
    s = FINAL['cohort_summaries'][vid]
    summary_rows.append({
        'variant': vid,
        'mean_day1_mae': s['day1_mae']['mean'],
        'mean_day1_rmse': s['day1_rmse']['mean'],
        'mean_pearson_r': s['residual_pearson_r']['mean'],
        'mean_std_ratio': s['residual_std_ratio']['mean'],
    })
pd.DataFrame(summary_rows)

## 5. Training and validation loss curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for hist, label, color in [(hist_r0, 'R0', 'C0'), (hist_r3, 'R3', 'C1')]:
    axes[0].plot(hist['epoch'], hist['loss'], label=f'{label} train', color=color)
    axes[0].plot(hist['epoch'], hist['val_loss'], '--', label=f'{label} val', color=color)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE loss')
axes[0].set_title('Training dynamics (optimisation focus)')
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

merged = eval_r0.merge(eval_r3, on='container_id', suffixes=('_R0', '_R3'))
axes[1].scatter(merged['day1_mae_R0'], merged['day1_mae_R3'], alpha=0.6, s=20)
lim = [0, max(merged['day1_mae_R0'].max(), merged['day1_mae_R3'].max()) * 1.05]
axes[1].plot(lim, lim, 'k--', lw=1)
axes[1].set_xlabel('R0 day1 MAE (%)')
axes[1].set_ylabel('R3 day1 MAE (%)')
axes[1].set_title('Per-container MAE: R3 vs R0')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Optimisation metrics

In [ ]:
opt = pd.DataFrame(FINAL['optimization_summaries']).T
opt[['best_epoch', 'epochs_run', 'best_val_loss', 'generalisation_gap', 'convergence_speed']]

## 7. Forecast metric distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
metrics = [
    ('day1_mae', 'Day-1 MAE (%)'),
    ('day1_rmse', 'Day-1 RMSE (%)'),
    ('residual_pearson_r', 'Residual Pearson r'),
    ('residual_std_ratio', 'Residual std ratio'),
]
for ax, (col, title) in zip(axes.ravel(), metrics):
    ax.boxplot([eval_r0[col].dropna(), eval_r3[col].dropna()], tick_labels=['R0', 'R3'])
    ax.set_title(title)
    ax.grid(alpha=0.3)
plt.suptitle('Cohort metric distributions')
plt.tight_layout()
plt.show()

## 8. Statistical tests (bootstrap + Wilcoxon)

In [ ]:
rows = []
for metric, block in COMP['forecast_metrics'].items():
    rows.append({
        'metric': metric,
        'mean_delta_R3_minus_R0': block['mean_delta'],
        'wilcoxon_p': block['wilcoxon']['pvalue'],
        'bootstrap_ci_lower': block['bootstrap']['ci_lower'],
        'bootstrap_ci_upper': block['bootstrap']['ci_upper'],
        'fraction_improved': block['fraction_improved'],
    })
pd.DataFrame(rows)

## 9. Case studies

In [ ]:
merged = eval_r0.merge(eval_r3, on='container_id', suffixes=('_R0', '_R3'))
merged['delta_mae'] = merged['day1_mae_R3'] - merged['day1_mae_R0']
best_r3 = merged.nsmallest(3, 'delta_mae')[['container_id', 'day1_mae_R0', 'day1_mae_R3', 'delta_mae']]
worst_r3 = merged.nlargest(3, 'delta_mae')[['container_id', 'day1_mae_R0', 'day1_mae_R3', 'delta_mae']]
print('Top R3 improvements:')
display(best_r3)
print('Top R3 regressions:')
display(worst_r3)

## 10. Prediction examples (case study containers)

In [ ]:
import pickle

def plot_case(cid, ax):
    cache_r0 = pickle.load(open(EXP_DIR / 'variants' / 'R0' / 'evaluation' / 'inference_cache.pkl', 'rb'))
    cache_r3 = pickle.load(open(EXP_DIR / 'variants' / 'R3' / 'evaluation' / 'inference_cache.pkl', 'rb'))
    actual = cache_r0[cid]['actual_day1_real']
    pred0 = cache_r0[cid]['day1_final_real']
    pred3 = cache_r3[cid]['day1_final_real']
    prophet = cache_r0[cid]['day1_prophet_real']
    t = np.arange(len(actual))
    ax.plot(t, actual, 'k-', label='Actual', lw=1.5)
    ax.plot(t, prophet, ':', color='gray', label='Prophet')
    ax.plot(t, pred0, '--', label='Hybrid R0')
    ax.plot(t, pred3, '-.', label='Hybrid R3')
    ax.set_title(f'{cid}')
    ax.set_xlabel('Step (15 min)')
    ax.set_ylabel('CPU %')
    ax.legend(fontsize=7)
    ax.grid(alpha=0.3)

case_ids = list(best_r3['container_id'].head(2)) + list(worst_r3['container_id'].head(1))
fig, axes = plt.subplots(len(case_ids), 1, figsize=(11, 3 * len(case_ids)))
if len(case_ids) == 1:
    axes = [axes]
for ax, cid in zip(axes, case_ids):
    plot_case(cid, ax)
plt.suptitle('Day-1 forecast case studies')
plt.tight_layout()
plt.show()

## 11. Research question answers

In [ ]:
for key, block in ANSWERS.items():
    print(f"{key}: {block['answer']}")
    print(f"  {block['evidence']}")
    print()

## 12. Consolidated conclusion

The full research arc — Stage 0 (representation diagnostics) and RRE v1.0 (scaling experiment) — is documented in:

`docs/hybrid_residual_investigation/CONCLUSION.md`

**Thesis statement:** The Prophet complement residual is best modelled in level form with global z-score normalisation. Velocity was rejected (Stage 0). Robust median/MAD scaling did not improve forecasts (RRE v1.0). Residual preprocessing is not the Hybrid bottleneck under the frozen protocol.

## 13. Discussion

**Interpretation:** If R3 and R0 produce similar accuracy but different optimisation curves, the experiment supports the hypothesis that scaling affects gradient conditioning without changing learnable temporal content. If R3 does not improve accuracy or convergence, the frozen Hybrid z-score baseline remains scientifically justified.

Negative results are valuable: they confirm that residual representation was not the bottleneck — consistent with HCERL, peak-aware, and window experiments.